### 加载数据

In [4]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor


training_data = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=ToTensor(),
)

test_data = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=ToTensor(),
)

training_data_loader = DataLoader(training_data, batch_size=64)


In [5]:
test_data_loader = DataLoader(test_data, batch_size=64)

In [7]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        # 展平层，多维输入张量展平成一维向量
        self.flatten = nn.Flatten()
        self.line_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.line_relu_stack(x)
        return logits

model = NeuralNetwork()

In [8]:
# 设置超参数
learn_rate = 1e-3
batch_size = 64
epochs = 5

In [9]:
loss_fn = nn.CrossEntropyLoss()

In [10]:
optimizer = torch.optim.SGD(model.parameters(), lr=learn_rate)

In [11]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [12]:
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(training_data_loader, model, loss_fn, optimizer)
    test_loop(test_data_loader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.301302  [   64/60000]
loss: 2.292477  [ 6464/60000]
loss: 2.269318  [12864/60000]
loss: 2.272250  [19264/60000]
loss: 2.246617  [25664/60000]
loss: 2.217169  [32064/60000]
loss: 2.228970  [38464/60000]
loss: 2.190315  [44864/60000]
loss: 2.185441  [51264/60000]
loss: 2.166764  [57664/60000]
Test Error: 
 Accuracy: 40.5%, Avg loss: 2.153457 

Epoch 2
-------------------------------
loss: 2.155806  [   64/60000]
loss: 2.158144  [ 6464/60000]
loss: 2.093755  [12864/60000]
loss: 2.120700  [19264/60000]
loss: 2.075835  [25664/60000]
loss: 2.006374  [32064/60000]
loss: 2.044396  [38464/60000]
loss: 1.959914  [44864/60000]
loss: 1.957750  [51264/60000]
loss: 1.906928  [57664/60000]
Test Error: 
 Accuracy: 60.0%, Avg loss: 1.897154 

Epoch 3
-------------------------------
loss: 1.919158  [   64/60000]
loss: 1.905785  [ 6464/60000]
loss: 1.778402  [12864/60000]
loss: 1.828607  [19264/60000]
loss: 1.734627  [25664/60000]
loss: 1.669164  [32064/600